# 04c — Trajectory Stitching
Assemble per-camera pose tracks into one global trajectory in world coordinates,
then export as parquet / CSV and BVH for 3D puppet playback.

**Prerequisites**
- Notebook 02b completed  → `unified_timeline.csv` on Drive
- Notebook 02c completed  → `H_cam{N}.npy` + `camera_layout.json` on Drive
- Notebook 04 completed   → pose parquet files on Drive

In [ ]:
# ===== CONFIGURATION =====

GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH      = "main"

DRIVE_ROOT   = "/content/drive/Shareddrives/3R.Data/1P.SPRIND.Data/POC/Behavior"

# Session to stitch (folder name under raw_videos)
SESSION_NAME = "260608.00000009"    # <-- SET THIS

# Input paths
TIMELINE_CSV   = f"{DRIVE_ROOT}/{SESSION_NAME}/calibration_output/unified_timeline.csv"
POSE_DIR       = f"{DRIVE_ROOT}/pose_outputs"
CALIB_DIR      = f"{DRIVE_ROOT}/calibration"

# Calibration output (from NB02b, inside the session folder)
CALIB_JSON     = f"{DRIVE_ROOT}/{SESSION_NAME}/calibration_output/calibration_result.json"

# Output paths
STITCH_OUTPUT  = f"{DRIVE_ROOT}/stitched_trajectories"

# Tracking parameters
CONFIDENCE_THRESHOLD = 0.5   # min mean keypoint likelihood to include a frame
SMOOTH               = True  # apply Savitzky-Golay smoothing


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os, sys

REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print("Repo ready.")


In [ ]:
!pip install --quiet pandas numpy pyarrow scipy opencv-python-headless matplotlib tqdm
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("Dependencies ready.")


---
## Step 1 — Verify inputs

In [ ]:
from pathlib import Path
import pandas as pd

print("Checking input files …")

# Timeline
timeline_path = Path(TIMELINE_CSV)
if timeline_path.exists():
    tl = pd.read_csv(timeline_path)
    print(f"  ✓ Timeline:  {len(tl)} entries  columns: {list(tl.columns)}")
    print(tl[["filename","camera","fps","frame_count","start_time_abs"]].head(8).to_string(index=False))
else:
    print(f"  ✗ Timeline not found: {timeline_path}")
    print("    → Run 02b_CrossCamera_Calibration.ipynb first")

print()

# Homographies
calib_path = Path(CALIB_DIR)
for cam in [1, 2, 3, 4]:
    hf = calib_path / f"H_cam{cam}.npy"
    status = "✓" if hf.exists() else "✗"
    print(f"  {status} H_cam{cam}.npy")

lf = calib_path / "camera_layout.json"
print(f"  {'✓' if lf.exists() else '✗'} camera_layout.json")

print()

# Pose parquets
pose_path = Path(POSE_DIR)
pose_files = list(pose_path.rglob("*.parquet")) if pose_path.exists() else []
print(f"  Pose parquet files: {len(pose_files)}")
for pf in pose_files[:8]:
    print(f"    {pf.relative_to(pose_path)}")
if len(pose_files) > 8:
    print(f"    … and {len(pose_files)-8} more")


---
## Step 2 — Run the stitcher

In [ ]:
from src.tracking.stitch import stitch_session, CONFIDENCE_THRESHOLD as DEFAULT_CT
import src.tracking.stitch as _stitch_mod

# Override the module-level threshold if user changed it
_stitch_mod.CONFIDENCE_THRESHOLD = CONFIDENCE_THRESHOLD

print(f"Stitching session: {SESSION_NAME}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"Smoothing: {SMOOTH}")
print()

trajectory = stitch_session(
    session_name   = SESSION_NAME,
    pose_dir       = POSE_DIR,
    homography_dir = CALIB_DIR,
    timeline_csv   = TIMELINE_CSV,
    output_dir     = STITCH_OUTPUT,
    calib_json     = CALIB_JSON,
    smooth         = SMOOTH,
    verbose        = True,
)

print("\nTrajectory columns:", list(trajectory.columns))
print()
print(trajectory[["abs_time_sec","camera","source","centroid_wx","centroid_wy","mean_likelihood"]].head(10).to_string(index=False))


---
## Step 3 — Quality visualisation

### 3a. Source coverage

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(3, 1, figsize=(16, 10))

# ── Coverage bar (source per frame) ──────────────────────────────────────
ax = axes[0]
source_codes = trajectory["source"].astype("category").cat.codes
cmap = plt.cm.get_cmap("tab10", trajectory["source"].nunique())
ax.scatter(trajectory["abs_time_sec"], np.zeros(len(trajectory)),
           c=source_codes, cmap=cmap, s=1, alpha=0.7)
ax.set_yticks([])
ax.set_xlabel("Wall-clock time (s)")
ax.set_title("Detection source per frame")
patches = [
    mpatches.Patch(color=cmap(i), label=src)
    for i, src in enumerate(trajectory["source"].cat.categories)
]
ax.legend(handles=patches, loc="upper right", fontsize=8, ncol=2)

# ── Confidence over time ─────────────────────────────────────────────────
ax = axes[1]
ax.plot(trajectory["abs_time_sec"], trajectory["mean_likelihood"],
        lw=0.6, alpha=0.8, color="#2166AC")
ax.axhline(CONFIDENCE_THRESHOLD, color="red", lw=1, ls="--",
           label=f"threshold={CONFIDENCE_THRESHOLD}")
ax.set_ylabel("Mean likelihood")
ax.set_xlabel("Wall-clock time (s)")
ax.set_title("Keypoint confidence over time")
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)

# ── Trajectory path in world coords ──────────────────────────────────────
ax = axes[2]
sc = ax.scatter(
    trajectory["centroid_wx"], trajectory["centroid_wy"],
    c=trajectory["abs_time_sec"], cmap="viridis",
    s=2, alpha=0.6
)
plt.colorbar(sc, ax=ax, label="Time (s)")
ax.set_xlabel("World X (cm) — direction of travel →")
ax.set_ylabel("World Y (cm)")
ax.set_title("Animal centroid path in world coordinates")
ax.set_aspect("equal")

plt.tight_layout()
fig_path = f"{STITCH_OUTPUT}/{SESSION_NAME}_trajectory_overview.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")


### 3b. Per-keypoint world path

In [ ]:
KEYPOINT_NAMES = [
    "snout", "left_ear", "right_ear", "neck",
    "shoulders", "mid_back", "hip", "tail_base",
]

n_kps = len(KEYPOINT_NAMES)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, kp in enumerate(KEYPOINT_NAMES):
    ax   = axes[i]
    wx_c = f"{kp}_wx"
    wy_c = f"{kp}_wy"
    if wx_c not in trajectory.columns:
        ax.text(0.5, 0.5, "N/A", ha="center", transform=ax.transAxes)
        ax.set_title(kp)
        continue
    ax.scatter(
        trajectory[wx_c].dropna(),
        trajectory[wy_c].dropna(),
        s=1, alpha=0.4, c=range(trajectory[wx_c].notna().sum()), cmap="plasma"
    )
    ax.set_title(kp, fontsize=10)
    ax.set_xlabel("X (cm)", fontsize=8)
    ax.set_ylabel("Y (cm)", fontsize=8)
    ax.set_aspect("equal")

plt.suptitle("Per-keypoint world paths", fontsize=13)
plt.tight_layout()
kp_path = f"{STITCH_OUTPUT}/{SESSION_NAME}_keypoint_paths.png"
plt.savefig(kp_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {kp_path}")


---
## Step 4 — Skeleton overlay on reference frame

Draws the stitched skeleton back onto the raw video frames for visual QC.
This helps confirm world → pixel reprojection is correct.

In [ ]:
import cv2, json, numpy as np
from pathlib import Path
from src.calibration.homography import load_all_homographies, world_to_pixel

homographies = load_all_homographies(CALIB_DIR)

SKELETON_EDGES = [
    ("tail_base", "hip"),
    ("hip",       "mid_back"),
    ("mid_back",  "shoulders"),
    ("shoulders", "neck"),
    ("neck",      "left_ear"),
    ("neck",      "right_ear"),
    ("neck",      "snout"),
]
KP_COLOUR  = (0, 230, 0)
BONE_COLOUR= (0, 180, 230)

# Pick a few frames spread across the trajectory for QC
sample_idx = np.linspace(0, len(trajectory)-1, 12, dtype=int)

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

import subprocess

for plot_i, traj_i in enumerate(sample_idx):
    row   = trajectory.iloc[traj_i]
    cam   = int(row.get("camera", 1))
    H     = homographies.get(cam)
    ax    = axes[plot_i]

    # Try to find the source video and decode the matching frame
    video_name = str(row.get("filename", ""))
    video_path = next(
        (p for p in Path(DRIVE_ROOT).rglob("*")
         if p.name == video_name), None
    )
    frame_idx = int(row.get("frame", 0))

    if video_path and H is not None:
        cmd = [
            "ffmpeg", "-y", "-i", str(video_path),
            "-vf", f"select=eq(n\\,{frame_idx})",
            "-vframes", "1",
            "-f", "image2pipe", "-pix_fmt", "rgb24", "-vcodec", "rawvideo", "pipe:1"
        ]
        res = subprocess.run(cmd, capture_output=True, timeout=30)
        if res.returncode == 0 and res.stdout:
            probe = subprocess.run(
                ["ffprobe", "-v", "quiet", "-print_format", "json",
                 "-show_streams", str(video_path)],
                capture_output=True, text=True, timeout=20
            )
            info = json.loads(probe.stdout)
            for s in info.get("streams", []):
                if s.get("codec_type") == "video":
                    w, h = int(s["width"]), int(s["height"])
                    frame = np.frombuffer(res.stdout, dtype=np.uint8).reshape(h, w, 3).copy()
                    # Project world keypoints back to pixel space
                    kp_pixels = {}
                    for kp in KEYPOINT_NAMES:
                        wx = row.get(f"{kp}_wx")
                        wy = row.get(f"{kp}_wy")
                        if wx is not None and not np.isnan(wx):
                            u, v = world_to_pixel(float(wx), float(wy), H)
                            kp_pixels[kp] = (int(u), int(v))
                    # Draw skeleton
                    for kp_a, kp_b in SKELETON_EDGES:
                        if kp_a in kp_pixels and kp_b in kp_pixels:
                            cv2.line(frame, kp_pixels[kp_a], kp_pixels[kp_b],
                                     BONE_COLOUR, 2)
                    for kp, (u, v) in kp_pixels.items():
                        cv2.circle(frame, (u, v), 5, KP_COLOUR, -1)
                    ax.imshow(frame)
                    ax.set_title(f"t={row['abs_time_sec']:.1f}s Cam{cam}", fontsize=8)
                    break

    ax.axis("off")

plt.suptitle("Skeleton reprojection QC — world coords → camera pixels", fontsize=12)
plt.tight_layout()
overlay_path = f"{STITCH_OUTPUT}/{SESSION_NAME}_skeleton_overlay.png"
plt.savefig(overlay_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved: {overlay_path}")


---
## Step 5 — Export BVH for 3D puppet

In [ ]:
from src.export.bvh_writer import write_bvh
import numpy as np

# Use the average FPS across cameras as the BVH frame rate
tl_session = pd.read_csv(TIMELINE_CSV)
if "session" in tl_session.columns:
    tl_session = tl_session[tl_session["session"] == SESSION_NAME]
bvh_fps = float(tl_session["fps"].mean()) if not tl_session.empty else 10.0
print(f"BVH frame rate: {bvh_fps:.2f} fps")

bvh_path = Path(STITCH_OUTPUT) / f"{SESSION_NAME}_trajectory.bvh"
write_bvh(
    trajectory  = trajectory,
    output_path = bvh_path,
    fps         = bvh_fps,
    verbose     = True,
)


---
## Step 6 — Summary of all outputs

In [ ]:
out_dir = Path(STITCH_OUTPUT)
print(f"Output directory: {out_dir}")
print()
total_bytes = 0
for f in sorted(out_dir.glob(f"{SESSION_NAME}*")):
    sz = f.stat().st_size
    total_bytes += sz
    print(f"  {f.name}  ({sz/1024:.0f} KB)")

print(f"\nTotal: {total_bytes/1024:.0f} KB")
print()
print("Files description:")
print("  *_stitched_trajectory.parquet  — full trajectory, all keypoints, world coords")
print("  *_stitched_trajectory.csv      — same, human-readable CSV")
print("  *_trajectory.bvh               — 3D skeleton animation for Blender/Maya")
print("  *_trajectory_overview.png      — QC plot: path + confidence + source")
print("  *_keypoint_paths.png           — per-keypoint world paths")
print("  *_skeleton_overlay.png         — reprojected skeleton on video frames")
